<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# RUL Prediction Model Performance Evaluation Notebook #
This notebook is meant as a way to evaluate training the RUL prediction model.  It will make use of MLFlow to track the metrics of the model.  It will first be used to baseline the performance of the RUL prediction model on the NASA CMAPSS datasets.

In [ ]:
import mlflow
import os
import sys
import pandas as pd
from mlflow.models import infer_signature

module_paths = [
    os.path.abspath(os.path.join('..')),
]
for module_path in module_paths:
    if module_path not in sys.path:
        sys.path.append(module_path)

from pdm.rul_model import RULModel
from pdm.config import RULConfig

***

### Instantiate the RUL Prediction Model ###
> First Experiment will use FD001 from the NASA CMAPSS Datasets

In [ ]:
filename = "FD001"
sensor_cols = ["s" + str(i) for i in set(range(1, 22))] + [
    "setting1",
    "setting2",
    "setting3",
]

save_file = "C:/models/"+filename+".h5"

cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, training_patience=10, attention_width=20)
print(cfg.model_dump())
model = RULModel(**cfg.model_dump())

### Set up MLflow to connect to a Local Instance ###

In [ ]:
# Set the tracking server uri for logging
# Don't forget to start the logging server!
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

### Load the Dataset for Training the model ###

In [ ]:
# Specify the file path
filepath = '../data/training/train_'+filename+'.txt'

# Set up Additional Parameters to be used in data exploration
# Maximum Remaining Useful Life value should be chosen for "no wear" on component.  This should also be chosen to leave enough time to order new component
max_RUL = 130

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

# Load the file into Pandas dataframe
train_data = pd.read_csv(filepath, sep=" ", header=None)
#train_data.drop(train_data.columns[[26, 27]], axis=1, inplace=True)
train_data.columns = column_names

In [ ]:
# Create a new MLflow Experiment
mlflow.set_experiment("RUL Prediction - CMAPSS "+filename)

mlflow.autolog(log_models=True, exclusive=True)
mlflow.tensorflow.autolog(log_models=True)

with mlflow.start_run() as run:
    
    mlflow.log_params(cfg.model_dump())
    
    mlflow.set_tag("Training Info", "Training RUL Model using the "+filename+" NASA CMAPSS Dataset")
    model.batch_train(train_data)
    
    # mlflow.tensorflow.log_model(
    #     model=model.model,    
    #     artifact_path="tensorflow-model",
    #     #signature=signature,
    #     registered_model_name="rul-model-"+filename
    # ) 
   
    
    

### Prepare the Test Data to make Predictions ###

In [ ]:
# Specify the file path
filepath = '../data/test/test_'+filename+'.txt'

# Set up Additional Parameters to be used in data exploration
# Maximum Remaining Useful Life value should be chosen for "no wear" on component.  This should also be chosen to leave enough time to order new component
max_RUL = 130

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

# Load the file into Pandas dataframe
test_data = pd.read_csv(filepath, sep=" ", header=None)
#test_data.drop(test_data.columns[[26, 27]], axis=1, inplace=True)
test_data.columns = column_names

### I would like to make predictions here and then compare to baseline models in MLflow ###

In [ ]:
# From https://mlflow.org/docs/latest/models.html#performing-model-validation
""" #Define criteria for model to be validated against
thresholds = {
    "accuracy_score": MetricThreshold(
        threshold=0.8,  # accuracy should be >=0.8
        min_absolute_change=0.05,  # accuracy should be at least 0.05 greater than baseline model accuracy
        min_relative_change=0.05,  # accuracy should be at least 5 percent greater than baseline model accuracy
        greater_is_better=True,
    ),
}

with mlflow.start_run() as run:
    candidate_model_uri = mlflow.sklearn.log_model(
        candidate_model, "candidate_model", signature=signature
    ).model_uri
    baseline_model_uri = mlflow.sklearn.log_model(
        baseline_model, "baseline_model", signature=signature
    ).model_uri

    mlflow.evaluate(
        candidate_model_uri,
        eval_data,
        targets="label",
        model_type="classifier",
        validation_thresholds=thresholds,
        baseline_model=baseline_model_uri,
    ) """